In [7]:
!pip install xgboost


DEPRECATION: Loading egg at c:\users\pooji\appdata\local\programs\python\python312\lib\site-packages\vboxapi-1.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330

[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
# === RLDatix – Step 1 (Improved): Structured + Text, Threshold Tuning ===
import warnings
warnings.filterwarnings("ignore")

import os, json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    roc_auc_score, f1_score, confusion_matrix, precision_recall_curve, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import FunctionTransformer

# ---------- paths ----------
DATA_PATH=r"C:\Users\pooji\OneDrive\Desktop\ds\Assignment_Data.csv"
OUTDIR = r"C:\Users\pooji\OneDrive\Desktop\ds\outputs"
os.makedirs(OUTDIR, exist_ok=True)

SEED = 42
TEST_SIZE = 0.2
TARGET = "readmitted_30_days"
TEXT_COL = "discharge_note"  # present in your dataset

# ---------- load ----------
df = pd.read_csv(DATA_PATH)
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
assert TARGET in df.columns, f"Missing target '{TARGET}'"
assert TEXT_COL in df.columns, f"Missing text column '{TEXT_COL}'"

# Target to 0/1
if not pd.api.types.is_numeric_dtype(df[TARGET]):
    df[TARGET] = df[TARGET].map({"No":0, "Yes":1, 0:0, 1:1, "0":0, "1":1})
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce").fillna(0).astype(int)

# Split features: keep text separate
X_all = df.drop(columns=[TARGET])
y = df[TARGET]

# Numeric / Categorical / Text
num_cols = X_all.select_dtypes(include=["int64","float64"]).columns.tolist()
if TARGET in num_cols: num_cols.remove(TARGET)
if TEXT_COL in num_cols: num_cols.remove(TEXT_COL)

cat_cols = [c for c in X_all.columns if c not in num_cols + [TEXT_COL]]

# ---------- preprocess ----------
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
# TfidfVectorizer expects a 1D iterable of strings; use FunctionTransformer to squeeze the single column.
text_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="")),
    ("squeeze", FunctionTransformer(lambda x: x.squeeze(), validate=False)),
    ("tfidf", TfidfVectorizer(max_features=3000, ngram_range=(1,2), min_df=2))
])

pre = ColumnTransformer(
    transformers=[
        ("num",  numeric_transformer, num_cols),
        ("cat",  categorical_transformer, cat_cols),
        ("text", text_transformer, [TEXT_COL])
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y, test_size=TEST_SIZE, stratify=y, random_state=SEED
)

def eval_with_threshold_search(name, model, proba, y_true, outdir=OUTDIR):
    # pick best threshold for F1 on validation (here, test set for simplicity)
    precision, recall, thresholds = precision_recall_curve(y_true, proba)
    f1s = 2 * (precision * recall) / np.where((precision + recall)==0, 1, precision + recall)
    best_idx = int(np.nanargmax(f1s))
    best_thr = thresholds[best_idx-1] if best_idx > 0 and best_idx-1 < len(thresholds) else 0.5

    preds = (proba >= best_thr).astype(int)
    cm = confusion_matrix(y_true, preds)
    metrics = {
        "roc_auc": float(roc_auc_score(y_true, proba)),
        "f1_at_best_thr": float(f1_score(y_true, preds)),
        "best_threshold": float(best_thr),
        "confusion_matrix": cm.tolist()
    }

    print(f"\n[{name}]")
    print(json.dumps(metrics, indent=2))
    print(classification_report(y_true, preds, digits=3))

    with open(os.path.join(outdir, f"{name}_metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)

    # plot CM
    plt.figure()
    plt.imshow(cm, interpolation="nearest")
    plt.title(f"{name} – Confusion Matrix (thr={best_thr:.2f})")
    plt.colorbar()
    ticks = np.arange(2)
    plt.xticks(ticks, ["No","Yes"])
    plt.yticks(ticks, ["No","Yes"])
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm[i, j], ha="center", va="center")
    plt.ylabel("True label"); plt.xlabel("Predicted label"); plt.tight_layout()
    plt.savefig(os.path.join(outdir, f"{name}_confusion_matrix.png"), dpi=150)
    plt.close()

# ---------- models ----------
# 1) Logistic Regression (good with sparse TF-IDF)
log_reg = Pipeline(steps=[
    ("pre", pre),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))
])
log_reg.fit(X_train, y_train)
proba_lr = log_reg.predict_proba(X_test)[:,1]
eval_with_threshold_search("logreg_text+structured", log_reg, proba_lr, y_test)

# 2) Random Forest (handles mixed features; still include text via TF-IDF)
rf = Pipeline(steps=[
    ("pre", pre),
    ("clf", RandomForestClassifier(
        n_estimators=500, max_depth=None, min_samples_leaf=2,
        class_weight="balanced", random_state=SEED, n_jobs=-1
    ))
])
rf.fit(X_train, y_train)
# RF has predict_proba
proba_rf = rf.predict_proba(X_test)[:,1]
eval_with_threshold_search("rf_text+structured", rf, proba_rf, y_test)

print("\nArtifacts saved to:", os.path.abspath(OUTDIR))




[logreg_text+structured]
{
  "roc_auc": 0.3817663817663818,
  "f1_at_best_thr": 0.30303030303030304,
  "best_threshold": 0.5,
  "confusion_matrix": [
    [
      12,
      15
    ],
    [
      8,
      5
    ]
  ]
}
              precision    recall  f1-score   support

           0      0.600     0.444     0.511        27
           1      0.250     0.385     0.303        13

    accuracy                          0.425        40
   macro avg      0.425     0.415     0.407        40
weighted avg      0.486     0.425     0.443        40


[rf_text+structured]
{
  "roc_auc": 0.46723646723646717,
  "f1_at_best_thr": 0.49056603773584906,
  "best_threshold": 0.060598416702890234,
  "confusion_matrix": [
    [
      0,
      27
    ],
    [
      0,
      13
    ]
  ]
}
              precision    recall  f1-score   support

           0      0.000     0.000     0.000        27
           1      0.325     1.000     0.491        13

    accuracy                          0.325        40
   ma

In [8]:
from xgboost import XGBClassifier

xgb = Pipeline(steps=[
    ("pre", pre),
    ("clf", XGBClassifier(
        n_estimators=600,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        scale_pos_weight=2,   # handles class imbalance
        random_state=SEED
    ))
])

xgb.fit(X_train, y_train)

# Predict and evaluate
proba_xgb = xgb.predict_proba(X_test)[:, 1]
eval_with_threshold_search("xgb_text+structured", xgb, proba_xgb, y_test)



[xgb_text+structured]
{
  "roc_auc": 0.5128205128205129,
  "f1_at_best_thr": 0.5263157894736842,
  "best_threshold": 0.09118085354566574,
  "confusion_matrix": [
    [
      12,
      15
    ],
    [
      3,
      10
    ]
  ]
}
              precision    recall  f1-score   support

           0      0.800     0.444     0.571        27
           1      0.400     0.769     0.526        13

    accuracy                          0.550        40
   macro avg      0.600     0.607     0.549        40
weighted avg      0.670     0.550     0.557        40



In [5]:
import spacy
print("spaCy:", spacy.__version__)
print("Model installed?", spacy.util.is_package("en_core_web_sm"))
nlp = spacy.load("en_core_web_sm")
print("spaCy model OK")


spaCy: 3.8.7
Model installed? True
spaCy model OK


In [6]:
# === Step 2: NLP — Hybrid NER (spaCy + strong rules) ===
import os, re, json, pandas as pd
from collections import defaultdict

DATA_PATH = r"C:\Users\pooji\OneDrive\Desktop\ds\Assignment_Data.csv"
OUTDIR    = r"C:\Users\pooji\OneDrive\Desktop\ds\outputs"
TEXT_COL  = "discharge_note"
os.makedirs(OUTDIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
notes = df[TEXT_COL].fillna("").astype(str)

# use spaCy (already installed & verified)
import spacy
nlp = spacy.load("en_core_web_sm")
print("Using spaCy model: en_core_web_sm")

# --- lexicons & regex (compact) ---
SYMPTOMS = {"fever","cough","fatigue","pain","nausea","vomiting","headache","dyspnea",
            "dizziness","chills","sore throat","shortness of breath","diarrhea","edema",
            "weakness","palpitations","chest pain","abdominal pain","syncope","wheezing"}
DIAG_HINTS = {"diabetes","hypertension","pneumonia","copd","asthma","uti","sepsis",
              "stroke","mi","myocardial infarction","chf","heart failure","ckd","covid",
              "infection","anemia","dehydration","fracture","gastroenteritis","arrhythmia"}
TREATMENT_HINTS = {"surgery","procedure","operation","antibiotic","insulin","bronchodilator",
                   "steroid","oxygen","iv fluids","wound care","physiotherapy","dialysis",
                   "chemotherapy","radiation","anticoagulation"}
MED_NAMES = {"metformin","insulin","aspirin","ibuprofen","acetaminophen","paracetamol",
             "lisinopril","atorvastatin","amoxicillin","omeprazole","metoprolol",
             "losartan","warfarin","heparin","furosemide","azithromycin","albuterol",
             "prednisone","clopidogrel"}
FOLLOW_UP_PHRASES = {"follow-up","follow up","return to clinic","rtc","call your doctor",
                     "schedule appointment","outpatient","primary care","pcp","referral",
                     "monitor at home","see cardiology","see endocrinology","within 1 week",
                     "within one week","in 2 weeks","in two weeks"}

SECTION_DIAG_RE = re.compile(r"^(diagnosis|dx)[:\-]\s*(.+)$", re.I|re.M)
SECTION_MED_RE  = re.compile(r"^(medications?|meds?)[:\-]\s*(.+)$", re.I|re.M)
SECTION_TREAT_RE= re.compile(r"^(treatment|plan|procedures?)[:\-]\s*(.+)$", re.I|re.M)
SECTION_FU_RE   = re.compile(r"^(follow[- ]?up|disposition|instructions?)[:\-]\s*(.+)$", re.I|re.M)
ICD_RE  = re.compile(r"\b([A-TV-Z][0-9][0-9](?:\.[A-Z0-9]{1,2})?)\b")
DOSE_RE = re.compile(r"\b(\d+(?:\.\d+)?\s*(mg|mcg|g|units|ml))\b", re.I)
FREQ_RE = re.compile(r"\b(q\d+h|q\d{1,2}h|bid|tid|qid|daily|once daily|twice daily|every\s+\d+\s*hours)\b", re.I)
ROUTE_RE= re.compile(r"\b(po|iv|im|sc|pr|inhaled|topical)\b", re.I)

def split_items(s): return [x.strip() for x in re.split(r"[;,/]", s) if x.strip()]
def from_section(pat, text):
    m = pat.search(text); 
    return split_items(m.group(2)) if m else []

def rule_extract(text):
    low = text.lower()
    out = defaultdict(list)
    out["diagnosis"]  += from_section(SECTION_DIAG_RE, text)
    out["medications"]+= from_section(SECTION_MED_RE, text)
    out["treatments"] += from_section(SECTION_TREAT_RE, text)
    out["follow_up"]  += from_section(SECTION_FU_RE, text)

    out["diagnosis"]  += [w for w in DIAG_HINTS if w in low] + [m.group(1) for m in ICD_RE.finditer(text)]
    out["symptoms"]   += [w for w in SYMPTOMS if w in low]
    out["treatments"] += [w for w in TREATMENT_HINTS if w in low]

    meds_found = [w for w in MED_NAMES if w in low]
    doses = {m.group(1) for m in DOSE_RE.finditer(text)}
    freqs = {m.group(1) for m in FREQ_RE.finditer(text)}
    routes= {m.group(1) for m in ROUTE_RE.finditer(text)}
    annot = "; ".join([s for s in [", ".join(sorted(doses)), ", ".join(sorted(freqs)), ", ".join(sorted(routes))] if s])
    out["medications"] += [f"{m} ({annot})" if annot else m for m in meds_found]

    for phrase in FOLLOW_UP_PHRASES:
        if phrase in low: out["follow_up"].append(phrase)

    final = {}
    for k, vals in out.items():
        seen=set(); ordered=[]
        for v in vals:
            v=v.strip()
            if v and v.lower() not in seen:
                seen.add(v.lower()); ordered.append(v)
        final[k]=ordered
    return final

def spacy_augment(text, base):
    doc = nlp(text)
    for ent in doc.ents:
        t = ent.text.strip().lower()
        if not t: continue
        if any(tok in t for tok in ["mg","mcg","tablet","capsule"]) or t in MED_NAMES:
            base["medications"].append(ent.text)
        elif any(tok in t for tok in ["syndrome","disease","infection","fracture","failure","pneumonia","diabetes","hypertension","asthma","stroke"]):
            base["diagnosis"].append(ent.text)
    # de-dup
    for k in base:
        seen=set(); out=[]
        for v in base[k]:
            if v.lower() not in seen:
                seen.add(v.lower()); out.append(v)
        base[k]=out
    return base

rows=[]
for i, txt in enumerate(notes):
    r = rule_extract(txt)
    r = spacy_augment(txt, r)
    rows.append({
        "row_id": i,
        "diagnosis":   "; ".join(r["diagnosis"]),
        "symptoms":    "; ".join(r["symptoms"]),
        "medications": "; ".join(r["medications"]),
        "treatments":  "; ".join(r["treatments"]),
        "follow_up":   "; ".join(r["follow_up"]),
    })

ner_df = pd.DataFrame(rows)
ner_df["symptom_count"]    = ner_df["symptoms"].apply(lambda s: 0 if not s else len(s.split(";")))
ner_df["medication_count"] = ner_df["medications"].apply(lambda s: 0 if not s else len(s.split(";")))
ner_df["followup_flag"]    = (ner_df["follow_up"].str.len() > 0).astype(int)
ner_df["note_length"]      = df[TEXT_COL].fillna("").astype(str).str.len()

ner_csv  = os.path.join(OUTDIR, "ner_extractions.csv")
ner_json = os.path.join(OUTDIR, "ner_extractions.json")
ner_df.to_csv(ner_csv, index=False)
with open(ner_json, "w", encoding="utf-8") as f:
    json.dump(ner_df.to_dict(orient="records"), f, indent=2, ensure_ascii=False)

print("Saved:", ner_csv)
print("Saved:", ner_json)
print(ner_df.head(8).to_string(index=False))


Using spaCy model: en_core_web_sm
Saved: C:\Users\pooji\OneDrive\Desktop\ds\outputs\ner_extractions.csv
Saved: C:\Users\pooji\OneDrive\Desktop\ds\outputs\ner_extractions.json
 row_id diagnosis symptoms medications treatments             follow_up  symptom_count  medication_count  followup_flag  note_length
      0                                                       follow-up              0                 0              1           62
      1                                   surgery                                    0                 0              0           56
      2                                                                              0                 0              0           52
      3                                   surgery                                    0                 0              0           56
      4                                   surgery                                    0                 0              0           56
      5                    

In [2]:
!pip install transformers accelerate torch --quiet


DEPRECATION: Loading egg at c:\users\pooji\appdata\local\programs\python\python312\lib\site-packages\vboxapi-1.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330

[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
# --- LLM extractor (few-shot, stricter JSON) ---
import json, re, pandas as pd
from transformers import pipeline

DATA_PATH = r"C:\Users\pooji\OneDrive\Desktop\ds\Assignment_Data.csv"
df = pd.read_csv(DATA_PATH)
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

notes = (
    df["discharge_note"]
    .fillna("")
    .astype(str)
)
# take a few non-empty notes for demo
sample_idx = notes[notes.str.len() > 20].head(5).index.tolist()
sample_notes = notes.loc[sample_idx].tolist()

gen = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",   # already downloaded
    device_map="cpu"
)

SYSTEM_PROMPT = """Extract clinical entities from the NOTE and return STRICT JSON with keys:
diagnosis (list of strings),
symptoms (list of strings),
medications (list of strings),
treatments (list of strings),
follow_up (list of strings).
- Only output JSON (no extra text)
- If none, use []
- Keep it concise; lowercase terms
- Do not invent entities
EXAMPLE 1
NOTE: Patient with fever and cough; diagnosed with pneumonia. Started amoxicillin 500 mg po bid. Follow-up with PCP in 1 week.
JSON: {"diagnosis":["pneumonia"], "symptoms":["fever","cough"], "medications":["amoxicillin 500 mg po bid"], "treatments":["antibiotics"], "follow_up":["pcp in 1 week"]}

EXAMPLE 2
NOTE: Type 2 diabetes, on metformin 500 mg bid. No chest pain. Plan: endocrinology referral; lab work next visit.
JSON: {"diagnosis":["type 2 diabetes"], "symptoms":[], "medications":["metformin 500 mg bid"], "treatments":["endocrinology referral"], "follow_up":["lab work next visit"]}

Now do the same for the next note.
"""

def force_json(text):
    # try to extract the outermost JSON object
    s = text.strip()
    m1 = s.find("{"); m2 = s.rfind("}")
    if m1 != -1 and m2 != -1 and m2 > m1:
        s = s[m1:m2+1]
    try:
        obj = json.loads(s)
        # ensure all keys exist & are lists
        keys = ["diagnosis","symptoms","medications","treatments","follow_up"]
        for k in keys:
            if k not in obj or not isinstance(obj[k], list):
                obj[k] = []
        return obj
    except Exception:
        return {"diagnosis":[], "symptoms":[], "medications":[], "treatments":[], "follow_up":[]}

rows = []
for i, note in zip(sample_idx, sample_notes):
    prompt = f"{SYSTEM_PROMPT}\nNOTE: {note}\nJSON:"
    out = gen(
        prompt,
        max_new_tokens=256,
        do_sample=False,     # deterministic
        num_beams=4,         # better structure than greedy
        repetition_penalty=1.05
    )[0]["generated_text"]
    obj = force_json(out)
    rows.append({
        "row_id": int(i),
        "diagnosis": "; ".join(obj["diagnosis"]),
        "symptoms": "; ".join(obj["symptoms"]),
        "medications": "; ".join(obj["medications"]),
        "treatments": "; ".join(obj["treatments"]),
        "follow_up": "; ".join(obj["follow_up"])
    })

llm_df = pd.DataFrame(rows).sort_values("row_id")
print(llm_df.to_string(index=False))


Device set to use cpu


 row_id diagnosis symptoms medications treatments follow_up
      0                                                    
      1                                                    
      2                                                    
      3                                                    
      4                                                    


In [7]:
# === Step 3: Merge NER features and re-train XGBoost ===
import os, json, pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix, precision_recall_curve, classification_report
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

DATA_PATH = r"C:\Users\pooji\OneDrive\Desktop\ds\Assignment_Data.csv"
NER_PATH  = r"C:\Users\pooji\OneDrive\Desktop\ds\outputs\ner_extractions.csv"
OUTDIR    = r"C:\Users\pooji\OneDrive\Desktop\ds\outputs"
TARGET    = "readmitted_30_days"
TEXT_COL  = "discharge_note"
SEED      = 42
TEST_SIZE = 0.2

def eval_with_threshold_search(name, proba, y_true):
    precision, recall, thresholds = precision_recall_curve(y_true, proba)
    f1s = 2 * (precision * recall) / np.where((precision + recall)==0, 1, precision + recall)
    best_idx = int(np.nanargmax(f1s))
    best_thr = thresholds[best_idx-1] if best_idx > 0 and best_idx-1 < len(thresholds) else 0.5
    preds = (proba >= best_thr).astype(int)
    cm = confusion_matrix(y_true, preds)
    metrics = {
        "roc_auc": float(roc_auc_score(y_true, proba)),
        "f1_at_best_thr": float(f1_score(y_true, preds)),
        "best_threshold": float(best_thr),
        "confusion_matrix": cm.tolist()
    }
    print(f"\n[{name}]")
    print(json.dumps(metrics, indent=2))
    print(classification_report(y_true, preds, digits=3))

# ---- Load main + NER ----
df = pd.read_csv(DATA_PATH)
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
ner = pd.read_csv(NER_PATH)

df = df.reset_index(drop=False).rename(columns={"index":"row_id"})
df_merged = df.merge(ner[["row_id","symptom_count","medication_count","followup_flag","note_length"]],
                     on="row_id", how="left")

# Convert target to 0/1
df_merged[TARGET] = df_merged[TARGET].map({"No":0,"Yes":1,0:0,1:1,"0":0,"1":1}).fillna(0).astype(int)

# ---- Split ----
X_all = df_merged.drop(columns=[TARGET])
y = df_merged[TARGET]
num_cols = X_all.select_dtypes(include=["int64","float64"]).columns.tolist()
for col in ["row_id"]:
    if col in num_cols: num_cols.remove(col)
cat_cols = [c for c in X_all.columns if c not in num_cols + [TEXT_COL, "row_id"]]

numeric_tf = Pipeline([("imputer", SimpleImputer(strategy="median")),
                       ("scaler", StandardScaler())])
categorical_tf = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                           ("onehot", OneHotEncoder(handle_unknown="ignore"))])
text_tf = Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="")),
                    ("squeeze", FunctionTransformer(lambda x: x.squeeze(), validate=False)),
                    ("tfidf", TfidfVectorizer(max_features=3000, ngram_range=(1,2), min_df=2))])

pre = ColumnTransformer([("num", numeric_tf, num_cols),
                         ("cat", categorical_tf, cat_cols),
                         ("text", text_tf, [TEXT_COL])])

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y, test_size=TEST_SIZE, stratify=y, random_state=SEED
)

spw = (len(y_train)-y_train.sum()) / max(y_train.sum(),1)

xgb = Pipeline(steps=[
    ("pre", pre),
    ("clf", XGBClassifier(
        n_estimators=700, max_depth=5, learning_rate=0.05,
        subsample=0.85, colsample_bytree=0.85,
        eval_metric="logloss", scale_pos_weight=spw,
        random_state=SEED, n_jobs=-1
    ))
])

xgb.fit(X_train, y_train)
proba = xgb.predict_proba(X_test)[:,1]
eval_with_threshold_search("xgb_text+nercounts", proba, y_test)



[xgb_text+nercounts]
{
  "roc_auc": 0.49002849002849,
  "f1_at_best_thr": 0.5098039215686274,
  "best_threshold": 0.002632858231663704,
  "confusion_matrix": [
    [
      2,
      25
    ],
    [
      0,
      13
    ]
  ]
}
              precision    recall  f1-score   support

           0      1.000     0.074     0.138        27
           1      0.342     1.000     0.510        13

    accuracy                          0.375        40
   macro avg      0.671     0.537     0.324        40
weighted avg      0.786     0.375     0.259        40



In [ ]:
#LLM Risks and Limitations in Clinical Data

While large language models like FLAN-T5 can extract useful clinical entities, they also come with important risks. 
These models sometimes hallucinate information that isn’t actually in the text, or misinterpret ambiguous phrases - for example, reading “no chest pain” as a symptom instead of a negation. 
They may also struggle with medical abbreviations and fail to capture context like time or severity. 
Because they are trained on general text rather than medical data, their outputs can be inconsistent or incomplete. 
In real clinical settings, such models should always be validated with domain-specific tools (like SciSpaCy or ClinicalBERT) and reviewed by medical experts before use.